# Drawing Interface — Inline Widget untuk Colab/Jupyter

Canvas drawing inline dengan support **multi-character + 0-9 + A-Z**.

**Prasyarat:** Model sudah trained (`../models/char_model.keras`).

In [ ]:
!pip install -q tensorflow pillow scipy

In [ ]:
import os
import io
import base64
import numpy as np
import tensorflow as tf
from PIL import Image
from scipy import ndimage
from IPython.display import HTML, display
import matplotlib.pyplot as plt

MODEL_PATH = '../models/char_model.keras'
if not os.path.exists(MODEL_PATH):
    MODEL_PATH = '../models/char_model.h5'

model = tf.keras.models.load_model(MODEL_PATH)
print('Model loaded.')

def index_to_char(idx):
    if 0 <= idx <= 9:
        return str(idx)
    return chr(ord('A') + idx - 10)

## Drawing Canvas

Tulis 1 atau lebih karakter di kanvas, klik **Send to Python**, lalu jalankan cell predict di bawah.

In [ ]:
canvas_html = """
<style>
.dr-container { font-family: -apple-system, sans-serif; background: #1e1e2e; color: #cdd6f4;
   padding: 16px; border-radius: 12px; max-width: 700px; }
.dr-container h3 { margin: 0 0 12px 0; color: #89b4fa; }
.dr-canvas { background: black; border-radius: 8px; cursor: crosshair; touch-action: none;
   border: 2px solid #313244; display: block; }
.dr-controls { display: flex; gap: 8px; margin-top: 8px; }
.dr-btn { padding: 8px 16px; border: none; border-radius: 6px; font-weight: 600;
   cursor: pointer; font-size: 13px; }
.dr-clear { background: #f38ba8; color: white; }
.dr-predict { background: #a6e3a1; color: #1e1e2e; }
.dr-status { margin-top: 8px; font-size: 13px; color: #a6adc8; }
</style>
<div class='dr-container'>
  <h3>🖊 Tulis 0-9 atau A-Z (multiple karakter OK)</h3>
  <canvas id='drCanvas' width='600' height='240' class='dr-canvas'></canvas>
  <div class='dr-controls'>
    <button class='dr-btn dr-clear' onclick='drClear()'>🗑 Clear</button>
    <button class='dr-btn dr-predict' onclick='drExport()'>📤 Send to Python</button>
  </div>
  <div class='dr-status' id='drStatus'>Tulis dulu, lalu klik Send</div>
</div>
<script>
(function(){
  const c = document.getElementById('drCanvas');
  const ctx = c.getContext('2d');
  ctx.fillStyle = 'black'; ctx.fillRect(0,0,c.width,c.height);
  ctx.strokeStyle = 'white'; ctx.fillStyle = 'white';
  ctx.lineCap = 'round'; ctx.lineJoin = 'round'; ctx.lineWidth = 22;
  let drawing = false, lx = 0, ly = 0;
  const getPos = (e) => {
    const r = c.getBoundingClientRect();
    const sx = c.width / r.width, sy = c.height / r.height;
    const cx = e.touches ? e.touches[0].clientX : e.clientX;
    const cy = e.touches ? e.touches[0].clientY : e.clientY;
    return { x: (cx - r.left) * sx, y: (cy - r.top) * sy };
  };
  const start = (e) => { e.preventDefault(); drawing = true; const p = getPos(e); lx = p.x; ly = p.y;
    ctx.beginPath(); ctx.arc(p.x, p.y, ctx.lineWidth/2, 0, Math.PI*2); ctx.fill(); };
  const move = (e) => { if(!drawing) return; e.preventDefault(); const p = getPos(e);
    ctx.beginPath(); ctx.moveTo(lx, ly); ctx.lineTo(p.x, p.y); ctx.stroke(); lx = p.x; ly = p.y; };
  const end = (e) => { if(e) e.preventDefault(); drawing = false; };
  c.addEventListener('mousedown', start); c.addEventListener('mousemove', move);
  c.addEventListener('mouseup', end); c.addEventListener('mouseleave', end);
  c.addEventListener('touchstart', start); c.addEventListener('touchmove', move);
  c.addEventListener('touchend', end);
  window.drClear = () => { ctx.fillStyle = 'black'; ctx.fillRect(0,0,c.width,c.height);
    ctx.fillStyle = 'white'; document.getElementById('drStatus').innerText = 'Cleared.'; };
  window.drExport = () => {
    window._lastCharImage = c.toDataURL('image/png');
    document.getElementById('drStatus').innerText = '✓ Image siap. Run cell predict di bawah.';
  };
})();
</script>
"""
display(HTML(canvas_html))

In [ ]:
# Set mode di sini: 'digits', 'letters', atau 'mixed'
MODE = 'mixed'

from google.colab.output import eval_js
data_url = eval_js('window._lastCharImage || ""')

if not data_url or ',' not in data_url:
    print('⚠️ Belum ada gambar. Tulis di canvas dan klik Send.')
else:
    # Decode
    img_bytes = base64.b64decode(data_url.split(',')[1])
    img = Image.open(io.BytesIO(img_bytes)).convert('L')
    arr = np.array(img)
    
    # Segment pakai connected components
    binary = (arr > 30).astype(np.uint8)
    binary = ndimage.binary_dilation(binary, iterations=2)
    labeled, n_feat = ndimage.label(binary)
    
    if n_feat == 0:
        print('Kanvas kosong')
    else:
        # Bounding boxes
        bboxes = []
        for i in range(1, n_feat + 1):
            ys, xs = np.where(labeled == i)
            if len(xs) < 50: continue
            bboxes.append((xs.min(), ys.min(), xs.max(), ys.max()))
        bboxes.sort(key=lambda b: b[0])
        
        # Predict tiap karakter
        chars, confs, char_imgs = [], [], []
        for x1, y1, x2, y2 in bboxes:
            pad = 8
            crop = img.crop((max(0,x1-pad), max(0,y1-pad), 
                             min(img.width,x2+pad), min(img.height,y2+pad)))
            crop.thumbnail((20, 20), Image.Resampling.LANCZOS)
            small = Image.new('L', (28, 28), 0)
            offset = ((28 - crop.width) // 2, (28 - crop.height) // 2)
            small.paste(crop, offset)
            
            small_arr = np.array(small).astype('float32') / 255.0
            x_input = small_arr.reshape(1, 28, 28, 1)
            probs = model.predict(x_input, verbose=0)[0]
            
            # Apply mode filter
            if MODE == 'digits':
                masked = probs.copy(); masked[10:] = 0
                idx = int(np.argmax(masked))
            elif MODE == 'letters':
                masked = probs.copy(); masked[:10] = 0
                idx = int(np.argmax(masked))
            else:
                idx = int(np.argmax(probs))
            
            chars.append(index_to_char(idx))
            confs.append(float(probs[idx]))
            char_imgs.append(small_arr)
        
        # Output
        result_str = ''.join(chars)
        avg_conf = np.mean(confs)
        
        print(f'🎯 Predicted: "{result_str}"')
        print(f'📊 Avg confidence: {avg_conf*100:.1f}%')
        print(f'📝 Mode: {MODE}\n')
        
        # Visualize
        n = len(chars)
        fig, axes = plt.subplots(1, n, figsize=(2*n, 2.5))
        if n == 1: axes = [axes]
        for ax, ch, conf, im in zip(axes, chars, confs, char_imgs):
            ax.imshow(im, cmap='gray')
            ax.set_title(f'"{ch}"\n{conf*100:.0f}%')
            ax.axis('off')
        plt.tight_layout()
        plt.show()